In [1]:
import os, json
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN_CS781_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: False
GPU count: 0


In [3]:
# from collections import Counter

def stream_json_array(filepath):
    decoder = json.JSONDecoder()

    with open(filepath, "r", encoding="utf-8") as f:
        buffer = ""
        pos = 0
        started = False

        while True:
            chunk = f.read(1024 * 1024) 
            if not chunk:
                break

            buffer += chunk

            while True:
                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1

                if not started:
                    if pos >= len(buffer):
                        break

                    if buffer[pos] == "[":
                        pos += 1
                        started = True
                    else:
                        raise ValueError("Expected JSON array")

                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1

                # End of JSON array
                if pos < len(buffer) and buffer[pos] == "]":
                    return

                try:
                    obj, end = decoder.raw_decode(buffer, pos)
                except json.JSONDecodeError:
                    # Need more data
                    break

                yield obj
                pos = end

                while pos < len(buffer) and buffer[pos].isspace():
                    pos += 1

                if pos < len(buffer) and buffer[pos] == ",":
                    pos += 1
                elif pos < len(buffer) and buffer[pos] == "]":
                    return

            # Keep only the unprocessed part
            buffer = buffer[pos:]
            pos = 0

In [4]:
import random, time, re, heapq, pickle, math
from collections import Counter, defaultdict
import numpy as np
from huggingface_hub import hf_hub_download, HfApi

REPO = "Exploration-Lab/CS-781-Capstone"

# ---------------- what to build ----------------
TOTAL_TOKENS = 150_000_000          # train + val together
VAL_TOKENS = 5_000_000
RATIOS = {                          # share of the TOTAL_TOKENS taken from each source (must sum to 1)
    "acts": 0.25,
    "high_courts": 0.25,
    "supreme_court": 0.25,
    "tribunals": 0.25,
}
assert abs(sum(RATIOS.values()) - 1.0) < 1e-9

# ---------------- tokenizer ----------------
VOCAB_SIZE = 40_000                 # final size: 256 bytes + merges + special tokens
LOWERCASE = False                   # keep the original casing
SPECIAL_TOKENS = ["<|endoftext|>"]
TOK_TRAIN_WORDS = 40_000_000        # words used to learn the BPE merges (split by RATIOS)

# ---------------- collection ----------------
TOK_PER_WORD = 1.35                 # rough guess, only used to decide how much text to collect
SAFETY = 1.25                       # collect 25% extra text so no source runs short
MIN_DOC_WORDS = 50
SEED = 14

TMP_ROOT = "/kaggle/temp"
try:
    os.makedirs(TMP_ROOT, exist_ok=True)
except Exception:
    TMP_ROOT = "/tmp"
RAW_DIR = f"{TMP_ROOT}/raw"
MIX_DIR = f"{TMP_ROOT}/mix"
OUT_DIR = "/kaggle/working"
for d in (RAW_DIR, MIX_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# ---------------- find the four training files in the HF repo ----------------
api = HfApi()
train_files = [f for f in api.list_repo_files(REPO, repo_type="dataset")
               if "Phase_1" in f and "Train" in f and f.endswith(".json")]
SOURCE_FILES = {}
for name in RATIOS:
    match = [f for f in train_files if name in os.path.basename(f)]
    assert len(match) == 1, f"could not find exactly one file for '{name}': {match}"
    SOURCE_FILES[name] = match[0]
print(SOURCE_FILES)


# ---------------- helpers ----------------
if "stream_json_array" not in globals():          # use your own reader if it already exists
    def stream_json_array(path, chunk_size=1 << 24):
        dec = json.JSONDecoder()
        ws = " \t\r\n,"
        with open(path, "r", encoding="utf-8") as f:
            buf = ""
            while "[" not in buf:
                chunk = f.read(chunk_size)
                if not chunk:
                    return
                buf += chunk
            pos = buf.index("[") + 1
            while True:
                while pos < len(buf) and buf[pos] in ws:
                    pos += 1
                if pos >= len(buf):
                    chunk = f.read(chunk_size)
                    if not chunk:
                        return
                    buf, pos = chunk, 0
                    continue
                if buf[pos] == "]":
                    return
                try:
                    obj, pos = dec.raw_decode(buf, pos)
                    yield obj
                except json.JSONDecodeError:
                    chunk = f.read(chunk_size)
                    if not chunk:
                        raise
                    buf, pos = buf[pos:] + chunk, 0


def get_text(rec):
    if isinstance(rec, str):
        return rec
    if "text" in rec:
        return rec["text"]
    strings = [v for v in rec.values() if isinstance(v, str)]
    return max(strings, key=len) if strings else ""


def collect_source(name, raw_path, word_quota, out_path, seed):
    """Stream the whole file once, keep every `stride`-th document (spread over the entire file,
    not just the beginning) until `word_quota` words are collected. Output: one JSON line per doc."""
    rng = random.Random(seed)
    size = os.path.getsize(raw_path)

    n = total_bytes = total_words = 0                       # estimate document size from the first 300 docs
    for rec in stream_json_array(raw_path):
        t = get_text(rec)
        n += 1
        total_bytes += len(t.encode("utf-8"))
        total_words += len(t.split())
        if n >= 300:
            break
    avg_bytes, avg_words = total_bytes / n, max(total_words / n, 1)
    est_docs = size / (avg_bytes * 1.02)
    docs_needed = word_quota / avg_words
    stride = max(1, int(0.85 * est_docs / docs_needed))     # 0.85: the estimate is slightly optimistic
    offset = rng.randrange(stride)

    words = kept = 0
    with open(out_path, "w", encoding="utf-8") as out:
        for idx, rec in enumerate(stream_json_array(raw_path)):
            if idx % stride != offset:
                continue
            t = get_text(rec)
            w = len(t.split())
            if w < MIN_DOC_WORDS:
                continue
            out.write(json.dumps({"t": t}, ensure_ascii=False) + "\n")
            words += w
            kept += 1
            if words >= word_quota:
                break
    print(f"  {name}: kept {kept:,} docs, {words:,} words (quota {word_quota:,}), stride {stride}")
    return words, kept

{'acts': 'Phase_1_Pretraining/Train/train_acts.json', 'high_courts': 'Phase_1_Pretraining/Train/train_high_courts.json', 'supreme_court': 'Phase_1_Pretraining/Train/train_supreme_court.json', 'tribunals': 'Phase_1_Pretraining/Train/train_tribunals.json'}


In [5]:
collected = {}
for name, fn in SOURCE_FILES.items():
    t0 = time.time()
    word_quota = int(RATIOS[name] * TOTAL_TOKENS / TOK_PER_WORD * SAFETY)
    print(f"[{name}] downloading {fn} ...")
    raw_path = hf_hub_download(REPO, fn, repo_type="dataset", local_dir=RAW_DIR)
    print(f"  raw file: {os.path.getsize(raw_path) / 1e9:.2f} GB")
    collected[name] = collect_source(name, raw_path, word_quota, f"{MIX_DIR}/{name}.jsonl", SEED)
    os.remove(raw_path)                                    # free the disk before the next source
    print(f"  done in {(time.time() - t0) / 60:.1f} min")

short = [n for n, (w, _) in collected.items() if w < RATIOS[n] * TOTAL_TOKENS / TOK_PER_WORD]
if short:
    print("WARNING: these sources are small and may not reach their share:", short)

[acts] downloading Phase_1_Pretraining/Train/train_acts.json ...


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

  raw file: 0.73 GB
  acts: kept 5,278 docs, 34,734,391 words (quota 34,722,222), stride 2
  done in 0.2 min
[high_courts] downloading Phase_1_Pretraining/Train/train_high_courts.json ...


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

  raw file: 1.32 GB
  supreme_court: kept 8,901 docs, 34,723,058 words (quota 34,722,222), stride 5
  done in 0.2 min
[tribunals] downloading Phase_1_Pretraining/Train/train_tribunals.json ...


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  raw file: 6.48 GB
  tribunals: kept 29,936 docs, 34,723,223 words (quota 34,722,222), stride 25
  done in 0.9 min


In [7]:
GPT2_PATTERN = (
    r"'s|'t|'re|'ve|'m|'ll|'d"
    r"| ?[^\W\d_]+"          # optional space + letters
    r"| ?\d+"                # optional space + digits
    r"| ?[^\s\w]+"           # optional space + punctuation / symbols
    r"|\s+(?!\S)"            # whitespace run not followed by a non-space
    r"|\s+"                  # remaining whitespace (newlines etc.)
)
PATTERN = re.compile(GPT2_PATTERN)

# ---- 1) count pre-tokens on a sample that follows RATIOS ----
t0 = time.time()
word_freq = Counter()
for name, share in RATIOS.items():
    quota, seen = int(share * TOK_TRAIN_WORDS), 0
    with open(f"{MIX_DIR}/{name}.jsonl", encoding="utf-8") as f:
        for line in f:
            text = json.loads(line)["t"]
            if LOWERCASE:
                text = text.lower()
            seen += len(text.split())
            for piece in PATTERN.findall(text):
                word_freq[piece.encode("utf-8")] += 1
            if seen >= quota:
                break
    print(f"  {name}: {seen:,} words used for the tokenizer")

before = len(word_freq)
word_freq = Counter({w: c for w, c in word_freq.items() if c >= 2})   # drop one-off strings (names, typos): saves RAM/time
print(f"unique pre-tokens: {before:,} -> {len(word_freq):,} after dropping singletons | {time.time() - t0:.0f}s")


# ---- 2) fast incremental BPE training ----
def train_bpe(word_freq, vocab_size, num_special, log_every=2000):
    n_merges = vocab_size - 256 - num_special
    assert n_merges > 0, "VOCAB_SIZE too small"
    t0 = time.time()

    words = [list(w) for w in word_freq]
    freqs = list(word_freq.values())

    pair_counts = defaultdict(int)
    where = defaultdict(set)
    for wi, w in enumerate(words):
        f = freqs[wi]
        for pair in zip(w, w[1:]):
            pair_counts[pair] += f
            where[pair].add(wi)

    heap = [(-c, p) for p, c in pair_counts.items()]
    heapq.heapify(heap)

    merges = []
    while len(merges) < n_merges and heap:
        neg_c, pair = heapq.heappop(heap)
        c = pair_counts.get(pair, 0)
        if c <= 0 or -neg_c != c:
            continue

        new_id = 256 + len(merges)
        merges.append((pair, new_id))
        a, b = pair
        touched = set()

        for wi in list(where[pair]):
            w = words[wi]
            f = freqs[wi]
            if not any(w[i] == a and w[i + 1] == b for i in range(len(w) - 1)):
                continue

            for p in zip(w, w[1:]):
                pair_counts[p] -= f
                touched.add(p)

            new_w, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                    new_w.append(new_id)
                    i += 2
                else:
                    new_w.append(w[i])
                    i += 1
            words[wi] = new_w

            for p in zip(new_w, new_w[1:]):
                pair_counts[p] += f
                where[p].add(wi)
                touched.add(p)

        del where[pair]
        pair_counts.pop(pair, None)
        touched.discard(pair)
        for p in touched:
            if pair_counts[p] > 0:
                heapq.heappush(heap, (-pair_counts[p], p))

        if len(merges) % log_every == 0:
            print(f"Merges: {len(merges):,}/{n_merges:,} | Frequency: {c:,} | {time.time() - t0:.0f}s")

    print(f"Done: {len(merges):,} merges in {time.time() - t0:.1f}s")
    return merges


merges = train_bpe(word_freq, VOCAB_SIZE, len(SPECIAL_TOKENS))

bpe_model = {
    "vocab_size": 256 + len(merges) + len(SPECIAL_TOKENS),
    "merges": merges,
    "special_tokens": SPECIAL_TOKENS,
    "lowercase": LOWERCASE,
    "pattern": GPT2_PATTERN,
}
TOKENIZER_V3_PATH = f"{OUT_DIR}/legal_bpe_v3.pkl"
with open(TOKENIZER_V3_PATH, "wb") as f:
    pickle.dump(bpe_model, f)
print("saved", TOKENIZER_V3_PATH, "| vocab size:", bpe_model["vocab_size"])
del word_freq

  acts: 10,003,198 words used for the tokenizer
  high_courts: 10,026,208 words used for the tokenizer
  supreme_court: 10,001,869 words used for the tokenizer
  tribunals: 10,000,958 words used for the tokenizer
unique pre-tokens: 349,556 -> 182,934 after dropping singletons | 34s
Merges: 2,000/39,743 | Frequency: 8,078 | 10s
Merges: 4,000/39,743 | Frequency: 2,928 | 11s
Merges: 6,000/39,743 | Frequency: 1,465 | 12s
Merges: 8,000/39,743 | Frequency: 883 | 13s
Merges: 10,000/39,743 | Frequency: 601 | 14s
Merges: 12,000/39,743 | Frequency: 433 | 14s
Merges: 14,000/39,743 | Frequency: 323 | 14s
Merges: 16,000/39,743 | Frequency: 255 | 15s
Merges: 18,000/39,743 | Frequency: 204 | 15s
Merges: 20,000/39,743 | Frequency: 167 | 15s
Merges: 22,000/39,743 | Frequency: 139 | 16s
Merges: 24,000/39,743 | Frequency: 117 | 16s
Merges: 26,000/39,743 | Frequency: 100 | 16s
Merges: 28,000/39,743 | Frequency: 87 | 16s
Merges: 30,000/39,743 | Frequency: 76 | 16s
Merges: 32,000/39,743 | Frequency: 67 | 16

In [8]:
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

from datasets import load_dataset


class LegalBPETokenizer:

    def __init__(self, model_path):

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])

        # pair -> rank (lower rank = learned earlier = merged first)
        self.merge_rank = {
            tuple(pair): rank
            for rank, (pair, _) in enumerate(self.merges)
        }

        # id -> bytes table (O(1) decode)
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]

        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")

        self._special_re = (
            re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
            if self.special_tokens else None
        )
        self._cache = {}

    def _encode_piece(self, piece_bytes):

        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached

        ids = list(piece_bytes)

        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break

            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):

        if self.lowercase:
            text = text.lower()

        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):

        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)

        if add_eos:
            ids.append(self.eos_id)

        return ids

    def decode(self, ids):

        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])

        return b"".join(chunks).decode("utf-8", errors="replace")


In [9]:
# needs the LegalBPETokenizer class (the class cell you already have); load the NEW pkl
tokenizer = LegalBPETokenizer(TOKENIZER_V3_PATH)
assert tokenizer.vocab_size < 65535, "ids must fit in uint16"

TRAIN_OUT = f"{OUT_DIR}/train_v3.bin"
VAL_OUT = f"{OUT_DIR}/val_v3.bin"
VAL_FRAC = VAL_TOKENS / TOTAL_TOKENS

rng = random.Random(SEED)
need = {n: int(RATIOS[n] * TOTAL_TOKENS) for n in RATIOS}
got = {n: 0 for n in RATIOS}
words_seen = {n: 0 for n in RATIOS}
exhausted = {n: False for n in RATIOS}
files = {n: open(f"{MIX_DIR}/{n}.jsonl", encoding="utf-8") for n in RATIOS}


def estimate_doc_tokens(path, n_docs=100):                 # rough average document length in tokens
    words = docs = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            words += len(json.loads(line)["t"].split())
            docs += 1
            if docs >= n_docs:
                break
    return max(words / max(docs, 1) * TOK_PER_WORD, 1.0)


avg_doc_tokens = {n: estimate_doc_tokens(f"{MIX_DIR}/{n}.jsonl") for n in RATIOS}
full_tokens = {n: 0 for n in RATIOS}
docs_done = {n: 0 for n in RATIOS}
train_count = val_count = 0
last_print = 0
t0 = time.time()

with open(TRAIN_OUT, "wb") as train_f, open(VAL_OUT, "wb") as val_f:
    while True:
        active = [n for n in RATIOS if got[n] < need[n] and not exhausted[n]]
        if not active:
            break
        # pick the next document's source in proportion to the NUMBER of documents each source still needs
        # (remaining tokens / average doc length) => sources are spread evenly through the whole file
        weights = [(need[m] - got[m]) / avg_doc_tokens[m] for m in active]
        n = rng.choices(active, weights=weights)[0]
        line = files[n].readline()
        if not line:
            exhausted[n] = True
            continue
        text = json.loads(line)["t"]
        ids = tokenizer.encode(text, add_eos=True)
        full_tokens[n] += len(ids)
        docs_done[n] += 1
        if docs_done[n] >= 20:
            avg_doc_tokens[n] = full_tokens[n] / docs_done[n]
        ids = ids[:need[n] - got[n]]
        words_seen[n] += len(text.split())
        arr = np.asarray(ids, dtype=np.uint16)
        if rng.random() < VAL_FRAC:
            arr.tofile(val_f)
            val_count += len(ids)
        else:
            arr.tofile(train_f)
            train_count += len(ids)
        got[n] += len(ids)

        total = train_count + val_count
        if total - last_print >= 5_000_000:
            last_print = total
            print(f"{total:,} tokens | {total / (time.time() - t0):,.0f} tok/s | {(time.time() - t0) / 60:.1f} min")

for f in files.values():
    f.close()

print("\nFinished!")
print(f"train tokens: {train_count:,} | val tokens: {val_count:,} | total: {train_count + val_count:,}")
for n in RATIOS:
    tpw = got[n] / max(words_seen[n], 1)
    flag = "" if got[n] >= 0.99 * need[n] else "   <-- SHORT: raise SAFETY and re-run the collection cell for this source"
    print(f"{n:>14}: {got[n]:>12,} tokens ({got[n] / max(train_count + val_count, 1):.1%}) | {tpw:.2f} tokens/word{flag}")

5,012,806 tokens | 1,140,890 tok/s | 0.1 min
10,021,233 tokens | 1,194,946 tok/s | 0.1 min
15,021,343 tokens | 1,208,951 tok/s | 0.2 min
20,047,849 tokens | 1,226,365 tok/s | 0.3 min
25,134,667 tokens | 1,226,333 tok/s | 0.3 min
30,137,224 tokens | 1,235,236 tok/s | 0.4 min
35,145,637 tokens | 1,222,979 tok/s | 0.5 min
40,146,215 tokens | 1,230,526 tok/s | 0.5 min
45,147,279 tokens | 1,236,574 tok/s | 0.6 min
50,147,587 tokens | 1,246,074 tok/s | 0.7 min
55,148,182 tokens | 1,255,194 tok/s | 0.7 min
60,158,088 tokens | 1,261,239 tok/s | 0.8 min
65,179,326 tokens | 1,262,568 tok/s | 0.9 min
70,183,238 tokens | 1,272,266 tok/s | 0.9 min
75,184,757 tokens | 1,278,967 tok/s | 1.0 min
80,203,936 tokens | 1,282,111 tok/s | 1.0 min
85,204,182 tokens | 1,282,725 tok/s | 1.1 min
90,209,754 tokens | 1,284,640 tok/s | 1.2 min
95,231,040 tokens | 1,279,278 tok/s | 1.2 min
100,238,296 tokens | 1,281,528 tok/s | 1.3 min
105,239,212 tokens | 1,280,182 tok/s | 1.4 min
110,239,795 tokens | 1,280,802 to